# 🧬 00 · 生物数据基础与序列表示

> 生物信息学的第一课：**了解数据长什么样**。
> 本节覆盖序列（DNA/RNA/蛋白）的字母表、FASTA/FASTQ 两大格式、
> Phred 质量值、GC 含量、k-mer 计数等最常用的基础操作。
> 全部手写实现 + 数值验证，不依赖 Biopython。

## 📋 本节要解决的问题

- DNA/RNA/蛋白序列的字母表与生物学对应关系？
- FASTA 与 FASTQ 的区别？Phred 质量值怎么换算？
- GC 含量为什么重要？怎么算？
- k-mer 是什么？在哪些地方用到？

## 1️⃣ 序列字母表：DNA / RNA / 蛋白质

| 类型 | 字母表 | 说明 |
|---|---|---|
| DNA | A C G T | 脱氧核苷酸，双链，A-T / C-G 配对 |
| RNA | A C G U | 尿嘧啶替代胸腺嘧啶 |
| 蛋白质 | 20 种氨基酸（A R N D C Q E G H I L K M F P S T W Y V） | 由三联密码子翻译 |

> **面试常问**：为什么 DNA 用 T、RNA 用 U？
> → U（尿嘧啶）是 T（胸腺嘧啶）脱甲基的产物，RNA 短命单链不需要 T 的修复优势。

In [1]:
# DNA 互补链（反向互补，5'->3' 惯例）
COMP = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}

def revcomp(seq):
    "返回反向互补序列"
    return ''.join(COMP.get(b, 'N') for b in reversed(seq.upper()))

dna = 'ATCGGCAT'
print('原始序列 :', dna)
print('反向互补 :', revcomp(dna))  # ATGCCGAT
assert revcomp(dna) == 'ATGCCGAT'

# GC 含量
def gc_content(seq):
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

p = gc_content('ATCGGC')
print('GC 含量(手算 4/6=0.667) =', round(p, 3))
assert abs(p - 4/6) < 1e-9
print('✅ 反向互补与 GC 含量正确')

原始序列 : ATCGGCAT
反向互补 : ATGCCGAT
GC 含量(手算 4/6=0.667) = 0.667
✅ 反向互补与 GC 含量正确


## 2️⃣ FASTA 格式解析

**FASTA**：以 `>` 开头的行为描述行，之后每行是序列（可跨多行）。
> 经典面试题：解析 FASTA 文件，把多行序列拼成一条。

In [2]:
def parse_fasta(text):
    "解析 FASTA 文本 -> {id: 序列}"
    records = {}
    cur_id = None
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            cur_id = line[1:].split()[0]
            records[cur_id] = ''
        else:
            records[cur_id] += line.upper()
    return records

fasta_text = '''>seq1 human gene X
ATCGGCAT
TTGCA
>seq2 mouse gene Y
GGCCATTAA'''
recs = parse_fasta(fasta_text)
print('记录数:', len(recs))
for k, v in recs.items():
    print(' ', k, '长度', len(v), '序列', v)
assert recs['seq1'] == 'ATCGGCATTTGCA'
assert recs['seq2'] == 'GGCCATTAA'
print('✅ FASTA 解析正确（跨行拼接，id 取首词）')

记录数: 2
  seq1 长度 13 序列 ATCGGCATTTGCA
  seq2 长度 9 序列 GGCCATTAA
✅ FASTA 解析正确（跨行拼接，id 取首词）


## 3️⃣ FASTQ 与 Phred 质量值

**FASTQ** 每条记录 4 行：`@id` / 序列 / `+` / 质量值字符串。
**Phred 质量值**：$Q = -10\log_{10}P$（碱基错误的概率）。
ASCII 编码：`chr(Q + 33)`（Sanger/Illumina 常见）。

> **面试**：Q30 意味着什么？→ $P=10^{-30/10}=0.001$，即千分之一出错概率；
> Q20=1%，Q30=0.1%。

In [3]:
def parse_fastq(text):
    "解析 FASTQ -> [(id, seq, qual)]"
    lines = text.strip().splitlines()
    out = []
    for i in range(0, len(lines), 4):
        rid = lines[i][1:]
        seq = lines[i+1]
        qual = lines[i+3]
        out.append((rid, seq, qual))
    return out

def phred_to_prob(q):
    return 10 ** (-q / 10)

def qual_to_phred_scores(qual_str):
    "ASCII 质量字符串 -> Phred 分数列表（Sanger +33）"
    return [ord(ch) - 33 for ch in qual_str]

fastq = '''@read1
ACGTACGT
+
IIIIIIII
@read2
TTTTAAAA
+
FFFFF'''
recs = parse_fastq(fastq)
print('记录数:', len(recs))
qs = qual_to_phred_scores(recs[0][2])
print('read1 质量分:', qs, '(I=73-33=40) → Q40')
print('Q40 的错误概率: %.5f' % phred_to_prob(40))
assert qs == [40]*8
print('✅ FASTQ 解析与 Phred 换算正确')

记录数: 2
read1 质量分: [40, 40, 40, 40, 40, 40, 40, 40] (I=73-33=40) → Q40
Q40 的错误概率: 0.00010
✅ FASTQ 解析与 Phred 换算正确


## 4️⃣ k-mer 计数（基因组与宏基因组的基石）

**k-mer**：长度为 k 的连续子串。应用：
- 基因组组装（de Bruijn 图）
- 序列比对索引（BLAST seed）
- 宏基因组物种识别（k-mer 指纹）
- 16S/全基因组距离估计（Jaccard of k-mer sets）

In [4]:
from collections import Counter

def get_kmers(seq, k):
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

def kmer_count(seq, k):
    return Counter(get_kmers(seq, k))

seq = 'ATCGATCGA'
cnt = kmer_count(seq, 3)
print('3-mer 计数:', dict(cnt))
print('总 k-mer 数 =', len(get_kmers(seq, 3)), '（n-k+1 =', len(seq)-3+1, '）')
assert cnt['ATC'] == 2 and cnt['TCG'] == 2
assert len(get_kmers(seq, 3)) == len(seq) - 3 + 1
print('✅ k-mer 计数正确')

3-mer 计数: {'ATC': 2, 'TCG': 2, 'CGA': 2, 'GAT': 1}
总 k-mer 数 = 7 （n-k+1 = 7 ）
✅ k-mer 计数正确


## 5️⃣ 手算示例：GC 含量与质量值转换

> **① GC 含量**：序列 `AAGGCC`（6bp，G+C = 2+2 = 4）→ GC = 4/6 = **0.667**
> **② Phred → 概率**：Q=30 → $P=10^{-30/10}=10^{-3}$ = **0.001**
> **③ ASCII → Q**：字符 `'5'` 的 ASCII=53 → Q = 53-33 = **20**（1% 错误率）

In [5]:
# 验证手算
seq = 'AAGGCC'
gc = (seq.count('G') + seq.count('C')) / len(seq)
print('GC(手算 4/6=0.667) =', round(gc, 3))
q30_p = 10 ** (-30/10)
print('Q30 概率(手算 1e-3) =', q30_p)
q_char = ord('5') - 33
print("'5' 的 Phred(手算 20) =", q_char)
assert abs(gc - 4/6) < 1e-9 and abs(q30_p - 0.001) < 1e-12 and q_char == 20
print('✅ 三步手算全部验证通过')

GC(手算 4/6=0.667) = 0.667
Q30 概率(手算 1e-3) = 0.001
'5' 的 Phred(手算 20) = 20
✅ 三步手算全部验证通过


## ✅ 自测清单

- [ ] 能说出 DNA/RNA/蛋白字母表与互补配对
- [ ] 能手写 FASTA/FASTQ 解析（含跨行拼接）
- [ ] 能换算 Phred 质量值与错误概率（Q30=0.1%）
- [ ] 能计算 GC 含量并解释生物学意义
- [ ] 能枚举 k-mer 并说明应用场景